In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import os
import pandas as pd

base_path = "../Daten"
sensors = ["Accelerometer", "Gyroscope", "Orientation"]

def trim_measurement(group, trim_seconds=3):
    min_time = group["seconds_elapsed"].min()
    max_time = group["seconds_elapsed"].max()
    return group.loc[
        (group["seconds_elapsed"] >= min_time + trim_seconds) &
        (group["seconds_elapsed"] <= max_time - trim_seconds)
    ]

raw_dfs = {sensor: [] for sensor in sensors}

measurement_id = 0

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    
    if os.path.isdir(folder_path):
        try:
            tag = pd.read_csv(os.path.join(folder_path, "Tags.csv"))
            
            for sensor in sensors:
                data = pd.read_csv(os.path.join(folder_path, f"{sensor}.csv"))
                data["Sensor"] = sensor
                data["time"] = pd.to_datetime(data["time"])
                data["Tag"] = tag["tag"].iloc[0]
                data["ID"] = measurement_id
                raw_dfs[sensor].append(data)
        
        except Exception as e:
            print(f"Fehler in Ordner {folder}: {e}")
        
        measurement_id += 1  # einmal pro Ordner, nicht pro Sensor

# Zusammenführen und trimmen
dfs = {}
for sensor in sensors:
    raw = pd.concat(raw_dfs[sensor], ignore_index=True)
    raw["ID_backup"] = raw["ID"]
    dfs[sensor] = (
        raw
        .groupby("ID_backup", group_keys=False)
        .apply(trim_measurement)
        .reset_index(drop=True)
    )

# Zugriff auf die einzelnen DataFrames
acc_df  = dfs["Accelerometer"]
gyr_df  = dfs["Gyroscope"]
ori_df  = dfs["Orientation"]

In [12]:
print("Accelerometer-Size:", acc_df.shape)
print("Gyroscope-Size:", gyr_df.shape)
print("Orientation-Size:", ori_df.shape)

Accelerometer-Size: (1860391, 8)
Gyroscope-Size: (1727291, 8)
Orientation-Size: (1822614, 12)


In [20]:
print("Accelerometer columns:", acc_df.columns)
print("Gyroscope columns:", gyr_df.columns)
print("Orientation columns:", ori_df.columns)

Accelerometer columns: Index(['time', 'seconds_elapsed', 'z', 'y', 'x', 'Sensor', 'Tag', 'ID'], dtype='str')
Gyroscope columns: Index(['time', 'seconds_elapsed', 'z', 'y', 'x', 'Sensor', 'Tag', 'ID'], dtype='str')
Orientation columns: Index(['time', 'seconds_elapsed', 'qz', 'qy', 'qx', 'qw', 'roll', 'pitch',
       'yaw', 'Sensor', 'Tag', 'ID'],
      dtype='str')


In [22]:
for name, df in [("Accelerometer", acc_df), ("Gyroscope", gyr_df), ("Orientation", ori_df)]:
    print(f"\n{name}:")
    summary = (
        df.groupby("ID")["seconds_elapsed"]
        .agg(
            Anzahl_Zeilen="count",
            Avg_Diff=lambda x: x.sort_values().diff().mean()
        )
        .reset_index()
    )
    summary["Frequenz_Hz"] = (1 / summary["Avg_Diff"]).round(1)
    print(summary)


Accelerometer:
      ID  Anzahl_Zeilen  Avg_Diff  Frequenz_Hz
0      0           3508  0.008673        115.3
1      1           1297  0.008673        115.3
2      2           1151  0.008673        115.3
3      3           1627  0.008673        115.3
4      4           3355  0.008673        115.3
..   ...            ...       ...          ...
556  556            853  0.008673        115.3
557  557            771  0.008702        114.9
558  558            680  0.008674        115.3
559  559            613  0.008681        115.2
560  560            530  0.008674        115.3

[561 rows x 4 columns]

Gyroscope:
      ID  Anzahl_Zeilen  Avg_Diff  Frequenz_Hz
0      0           3508  0.008673        115.3
1      1           1298  0.008673        115.3
2      2           1151  0.008673        115.3
3      3           1627  0.008673        115.3
4      4           3355  0.008673        115.3
..   ...            ...       ...          ...
556  556            853  0.008673        115.3
557  557

Notes:

- zuerst ein Dataframe erstellen
- Zeitstempel neu erstellen und aggregieren für evt. 60Hz, und durchschnitt nehmen
- 3D plot machen für eine Bewegung für verständnis
- Achsen vergleichen von Accelerometer und Gyroscope evt. überlappen
- oder nur mit Statistik Werten arbeiten (mean, median, min, max, q1, q2 etc.)

## Vorgehen Merging

Da die drei Sensoren teilweise unterschiedliche Aufnahmefrequenzen haben, besitzen sie unterschiedlich viele Zeilen pro Aufnahme. Um alle Sensoren in einen gemeinsamen Datensatz zusammenzuführen, wird ein einheitliches Zeitraster von 0.02s (50Hz) verwendet.

Dabei wird die Zeit jeder Aufnahme (ID) auf 0 normiert, sodass jede Aufnahme bei t=0 beginnt. Anschliessend werden alle Messwerte innerhalb eines 0.02s-Fensters zum Mittelwert aggregiert (Downsampling von ~115Hz auf 50Hz). Da Accelerometer und Gyroscope beide Spalten mit den Namen x, y und z besitzen, werden diese vor dem Zusammenführen umbenannt (z.B. x_acc, x_gyr), um Verwechslungen zu vermeiden.

In [27]:
def resample_sensor(df, sensor_cols, freq="20ms"):
    result = []
    for id_, group in df.groupby("ID"):
        tag = group["Tag"].iloc[0]
        # Auf 0 normieren
        elapsed_normalized = group["seconds_elapsed"] - group["seconds_elapsed"].min()
        group = group.set_index(
            pd.to_timedelta(elapsed_normalized, unit="s")
        )[sensor_cols]
        group = group.resample(freq).mean()
        group["ID"] = id_
        group["Tag"] = tag
        result.append(group)
    return pd.concat(result).reset_index(names="time_elapsed")

acc_rs  = resample_sensor(acc_df,  ["x", "y", "z"])
gyr_rs  = resample_sensor(gyr_df,  ["x", "y", "z"])
ori_rs  = resample_sensor(ori_df,  ["qx", "qy", "qz", "qw", "roll", "pitch", "yaw"])

# Spalten umbenennen vor dem Merge
acc_rs = acc_rs.rename(columns={"x": "x_acc", "y": "y_acc", "z": "z_acc"})
gyr_rs = gyr_rs.rename(columns={"x": "x_gyr", "y": "y_gyr", "z": "z_gyr"})

# Zusammenführen
merged_df = (
    acc_rs
    .merge(gyr_rs, on=["time_elapsed", "ID", "Tag"])
    .merge(ori_rs,  on=["time_elapsed", "ID", "Tag"])
)

print(merged_df.shape)
print(merged_df.head())

(800706, 16)
            time_elapsed     x_acc     y_acc     z_acc  ID     Tag     x_gyr  \
0        0 days 00:00:00  7.220327 -2.509777 -3.261837   0  Laufen  1.899331   
1 0 days 00:00:00.020000  5.171095 -5.565558 -3.215115   0  Laufen  1.741423   
2 0 days 00:00:00.040000  0.501864 -4.602707 -3.856779   0  Laufen  1.323591   
3 0 days 00:00:00.060000 -4.842717 -0.523853 -2.772057   0  Laufen  0.839685   
4 0 days 00:00:00.080000 -4.852034  1.710299 -0.924968   0  Laufen  0.579558   

      y_gyr     z_gyr        qx        qy        qz        qw      roll  \
0  0.997033  0.691957 -0.719674 -0.113393  0.076918  0.680626 -2.513721   
1  1.155756  0.363006 -0.709208 -0.105267  0.073267  0.693227 -2.155226   
2  1.328631  0.159894 -0.705078 -0.098675  0.069639  0.698770 -1.904056   
3  1.260112  0.456978 -0.700827 -0.080140  0.065243  0.705789 -1.292998   
4  0.564439  0.777784 -0.698070 -0.068643  0.069219  0.709358 -0.076087   

      pitch       yaw  
0  1.494964  2.243559  
1  1.52

In [26]:
print(merged_df["ID"].nunique())

561
